# SoftMeta Chatterbox TTS Server

Reliable personal Colab build for Chatterbox TTS, optional MOSS Generate Voice, and realistic EchoMimicV3 Flash Avatar Video.

- **Run all** starts the normal Chatterbox server quickly.
- MOSS and EchoMimicV3 Flash are optional because their separate environments and model downloads make setup slower.
- EchoMimicV3 Flash uses a 1.3B base model and 8-step TeaCache inference. The first avatar installation needs at least 28 GB free disk.
- An A100 accelerates inference, but it does not accelerate `apt`, `pip`, GitHub cloning, or Hugging Face downloads.
- This notebook clones the `main` branch directly. Upload the supplied GitHub files before running it.


In [ ]:
# Setup choices
# Keep both optional runtimes False for the fastest 3-5 minute TTS startup.
# Enable only the feature you need, then rerun the optional installer cell,
# the verification cell, and the server-start cell.

INSTALL_MOSS_RUNTIME = False  # @param {type:"boolean"}
INSTALL_AVATAR_RUNTIME = False  # @param {type:"boolean"}
FORCE_REINSTALL = False  # @param {type:"boolean"}

import os

os.environ["SOFTMETA_INSTALL_MOSS"] = "1" if INSTALL_MOSS_RUNTIME else "0"
os.environ["SOFTMETA_INSTALL_AVATAR"] = "1" if INSTALL_AVATAR_RUNTIME else "0"
os.environ["SOFTMETA_FORCE_REINSTALL"] = "1" if FORCE_REINSTALL else "0"
os.environ["MPLBACKEND"] = "Agg"

print("Install MOSS runtime:", INSTALL_MOSS_RUNTIME)
print("Install Avatar runtime:", INSTALL_AVATAR_RUNTIME)
print("Force reinstall:", FORCE_REINSTALL)


## Install the main Chatterbox server

This cell is idempotent. Rerunning it reuses the existing environment instead of deleting and rebuilding everything.


In [ ]:
%%bash
set -euo pipefail

export PIP_CACHE_DIR=/content/.cache/pip
missing_system=0
for command_name in ffmpeg git git-lfs curl lsof sox; do
  command -v "$command_name" >/dev/null 2>&1 || missing_system=1
done
if [[ "$missing_system" == "1" ]]; then
  apt-get update -qq
  apt-get install -y -qq \
    ffmpeg libsndfile1 git git-lfs curl ca-certificates lsof \
    sox libsox-fmt-all build-essential
else
  echo "System packages are already available."
fi

echo "GPU assigned by Colab:"
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true

mkdir -p /content/bin /content/.cache/pip
MICROMAMBA=/content/bin/micromamba
MICROMAMBA_VERSION=2.6.2-1
MICROMAMBA_URL="https://github.com/mamba-org/micromamba-releases/releases/download/${MICROMAMBA_VERSION}/micromamba-linux-64"

if [[ ! -x "$MICROMAMBA" ]]; then
  curl --fail --location --retry 5 --retry-delay 2 --retry-all-errors \
    --connect-timeout 30 "$MICROMAMBA_URL" --output "$MICROMAMBA"
  chmod +x "$MICROMAMBA"
fi
"$MICROMAMBA" --version

has_env() {
  "$MICROMAMBA" env list | awk '{print $1}' | grep -qx "$1"
}

if [[ "${SOFTMETA_FORCE_REINSTALL:-0}" == "1" ]] && has_env sm311; then
  "$MICROMAMBA" env remove -n sm311 -y || true
fi
if ! has_env sm311; then
  "$MICROMAMBA" create -y -n sm311 -c conda-forge python=3.11 pip
fi

cd /content

if [[ -d chatterbox-v2/.git ]]; then
  git -C chatterbox-v2 fetch --depth 1 origin tag v0.2.1
  git -C chatterbox-v2 checkout --detach v0.2.1
  git -C chatterbox-v2 reset --hard v0.2.1
else
  rm -rf chatterbox-v2
  git clone --branch v0.2.1 --depth 1 https://github.com/soft-meta/chatterbox-v2.git
fi

SERVER_REPO=/content/SoftMeta-Chatterbox-Repository
if [[ -d "$SERVER_REPO/.git" ]]; then
  git -C "$SERVER_REPO" fetch --depth 1 origin main
  git -C "$SERVER_REPO" checkout main
  git -C "$SERVER_REPO" reset --hard origin/main
else
  git clone --branch main --depth 1 https://github.com/soft-meta/Chatterbox-TTS-Server.git "$SERVER_REPO"
fi

# The GitHub repository currently contains both old root files and a newer
# uploaded folder. Select the newest complete SoftMeta server automatically.
PROJECT_SOURCE=$(python3 - <<'PYPROJECT'
import re
from pathlib import Path

repo = Path('/content/SoftMeta-Chatterbox-Repository')
candidates = []
for server_file in repo.glob('**/server.py'):
    project = server_file.parent
    if len(project.relative_to(repo).parts) > 3:
        continue
    required = [project / 'requirements-colab.txt', project / 'start.py',
                project / 'ui/index.html', project / 'scripts/install_echomimic_v3_flash_a100.sh']
    if not all(path.exists() for path in required):
        continue
    match = re.search(r'APP_VERSION\s*=\s*"([0-9.]+)"',
                      server_file.read_text(encoding='utf-8', errors='replace'))
    if not match:
        continue
    version = tuple(int(part) for part in match.group(1).split('.'))
    candidates.append((version, -len(project.relative_to(repo).parts), project))
if not candidates:
    raise SystemExit('No complete SoftMeta server folder was found in the GitHub repository.')
print(max(candidates)[2])
PYPROJECT
)
echo "Selected server source: $PROJECT_SOURCE"
printf '%s\n' "$PROJECT_SOURCE" > /content/softmeta_project_path.txt

MM=/content/bin/micromamba

if ! "$MM" run -n sm311 python - <<'PYCHECK' >/dev/null 2>&1
import torch
import torchaudio
import chatterbox
import perth
PYCHECK
then
  "$MM" run -n sm311 python -m pip install -U pip wheel
  "$MM" run -n sm311 python -m pip install "setuptools==80.9.0"
  "$MM" run -n sm311 python -m pip install \
    --index-url https://download.pytorch.org/whl/cu124 \
    torch==2.6.0 torchaudio==2.6.0
  "$MM" run -n sm311 python -m pip install --no-cache-dir chatterbox-tts==0.1.7
fi

"$MM" run -n sm311 python -m pip install --no-deps -e /content/chatterbox-v2
"$MM" run -n sm311 python -m pip install -r "$PROJECT_SOURCE/requirements-colab.txt"
"$MM" run -n sm311 python -m pip install "setuptools==80.9.0"

# Compile with the same Python 3.11 used by the server. This catches syntax
# errors before Uvicorn starts.
"$MM" run -n sm311 python -m compileall -q "$PROJECT_SOURCE"
SOFTMETA_PROJECT_SOURCE="$PROJECT_SOURCE" "$MM" run -n sm311 python - <<'PYIMPORT'
import os
import sys
sys.path.insert(0, os.environ['SOFTMETA_PROJECT_SOURCE'])
import avatar_engine
import server
print("Server source import check passed.")
PYIMPORT

echo "Main Chatterbox server environment is ready."


## Optional: Install MOSS Generate Voice

This cell is skipped unless `INSTALL_MOSS_RUNTIME` is enabled above. The first installation is large. Reruns reuse a verified environment.


In [ ]:
%%bash
set -euo pipefail
if [[ "${SOFTMETA_INSTALL_MOSS:-0}" != "1" ]]; then
  echo "Skipped MOSS runtime. Enable INSTALL_MOSS_RUNTIME only when you need Generate Voice."
  exit 0
fi

export SOFTMETA_SERVER_DIR="$(cat /content/softmeta_project_path.txt)"
export SOFTMETA_MOSS_DIR=/content/MOSS-TTS
export SOFTMETA_MOSS_ENV=moss312
export SOFTMETA_FORCE_REINSTALL="${SOFTMETA_FORCE_REINSTALL:-0}"

bash "$SOFTMETA_SERVER_DIR/scripts/install_moss_a100.sh" /content/bin/micromamba


## Optional: Install EchoMimicV3 Flash Generate Video

This cell is skipped unless `INSTALL_AVATAR_RUNTIME` is enabled above. The first run installs the pinned EchoMimicV3 Flash 8-step runtime and downloads its verified model components. Allow at least 28 GB of free disk.


In [ ]:
%%bash
set -euo pipefail
if [[ "${SOFTMETA_INSTALL_AVATAR:-0}" != "1" ]]; then
  echo "Skipped Avatar runtime. Enable INSTALL_AVATAR_RUNTIME only when you need Generate Video."
  exit 0
fi

export SOFTMETA_SERVER_DIR="$(cat /content/softmeta_project_path.txt)"
export SOFTMETA_ECHOMIMIC_DIR=/content/EchoMimicV3
export SOFTMETA_ECHOMIMIC_MODELS=/content/echomimic_v3_models
export SOFTMETA_AVATAR_ENV=echo310
export SOFTMETA_FORCE_REINSTALL="${SOFTMETA_FORCE_REINSTALL:-0}"
export MPLBACKEND=Agg

bash "$SOFTMETA_SERVER_DIR/scripts/install_echomimic_v3_flash_a100.sh" /content/bin/micromamba


## Verify installed runtimes

The main server is always verified. Optional runtimes are verified only when they exist.


In [ ]:
%%bash
set -euo pipefail
export MPLBACKEND=Agg
MM=/content/bin/micromamba

"$MM" run -n sm311 python - <<'PYMAIN'
import sys
from importlib.metadata import version
import torch
import torchaudio
import chatterbox
import perth
from softmeta_chatterbox import SoftMetaChatterboxEngine

print("Main Chatterbox environment")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)
print("Transformers:", version("transformers"))
if not torch.cuda.is_available():
    raise SystemExit("CUDA is unavailable. Select a GPU runtime.")
print("GPU:", torch.cuda.get_device_name(0))
print("Official Chatterbox package:", chatterbox.__file__)
print("PerTh watermarker callable:", callable(getattr(perth, "PerthImplicitWatermarker", None)))
print("SoftMeta engine adapter:", SoftMetaChatterboxEngine.__name__)
PYMAIN

if "$MM" env list | awk '{print $1}' | grep -qx moss312; then
  "$MM" run -n moss312 python - <<'PYMOSS'
import torch
import torchaudio
import soundfile
from transformers import AutoModel, AutoProcessor
from speechbrain.inference.speaker import EncoderClassifier
if not torch.cuda.is_available():
    raise SystemExit("CUDA is unavailable in the MOSS environment.")
print("MOSS runtime: ready")
print("MOSS GPU:", torch.cuda.get_device_name(0))
print("MOSS PyTorch:", torch.__version__)
print("MOSS TorchAudio:", torchaudio.__version__)
print("Speaker checker:", EncoderClassifier.__name__)
PYMOSS
else
  echo "MOSS runtime: not installed (Generate Voice remains optional)."
fi

if "$MM" env list | awk '{print $1}' | grep -qx echo310; then
  "$MM" run -n echo310 python - <<'PYAVATAR'
from pathlib import Path
import os
import sys
import torch
import transformers
import diffusers
import librosa
import moviepy
import pyloudnorm

root = Path("/content/EchoMimicV3")
models = Path("/content/echomimic_v3_models/flash")
required = [
    root / "infer_flash.py",
    root / ".softmeta_echomimic_v3_flash_v110",
    models / "Wan2.1-Fun-V1.1-1.3B-InP/diffusion_pytorch_model.safetensors",
    models / "Wan2.1-Fun-V1.1-1.3B-InP/Wan2.1_VAE.pth",
    models / "Wan2.1-Fun-V1.1-1.3B-InP/models_t5_umt5-xxl-enc-bf16.pth",
    models / "chinese-wav2vec2-base/config.json",
    models / "transformer/diffusion_pytorch_model.safetensors",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise SystemExit("Missing EchoMimicV3 files: " + ", ".join(missing))
sys.path.insert(0, str(root))
from src.wan_transformer3d_audio_2512 import WanTransformerAudioMask3DModel
if not torch.cuda.is_available():
    raise SystemExit("CUDA is unavailable in the Avatar environment.")
print("Avatar runtime: EchoMimicV3 Flash ready")
print("Avatar GPU:", torch.cuda.get_device_name(0))
print("Avatar VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 1))
print("EchoMimic Flash model:", WanTransformerAudioMask3DModel.__name__)
PYAVATAR
else
  echo "Avatar runtime: not installed (Generate Video remains optional)."
fi

echo "Runtime verification finished."


## Start SoftMeta

After installing an optional runtime later, rerun the verification cell and this server cell.


In [ ]:
import os
import signal
import socket
import subprocess
import time
from pathlib import Path
from IPython.display import HTML, display

PORT = 8004
PROJECT = Path("/content/softmeta_project_path.txt").read_text().strip()
PROJECT = Path(PROJECT)
LOG = Path("/content/softmeta_chatterbox.log")
PID_FILE = Path("/content/softmeta_chatterbox.pid")
MM = "/content/bin/micromamba"

if PID_FILE.exists():
    try:
        os.kill(int(PID_FILE.read_text().strip()), signal.SIGTERM)
        time.sleep(1)
    except Exception:
        pass
subprocess.run(f"lsof -t -i:{PORT} | xargs -r kill -9", shell=True, check=False)
LOG.unlink(missing_ok=True)

def env_python(name: str) -> str | None:
    try:
        return subprocess.check_output(
            [MM, "run", "-n", name, "python", "-c", "import sys; print(sys.executable)"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except subprocess.CalledProcessError:
        return None

env = {
    **os.environ,
    "PYTHONUNBUFFERED": "1",
    "MPLBACKEND": "Agg",
    "HF_HOME": "/content/hf_home",
    "HF_HUB_CACHE": "/content/hf_home/hub",
    "TRANSFORMERS_CACHE": "/content/hf_home/transformers",
    "SOFTMETA_DEVICE": "cuda",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:128",
    "SOFTMETA_AVATAR_GPU_PROFILE": "a100_40gb",
    "SOFTMETA_MODEL": "chatterbox",
    "SOFTMETA_MOSS_MODEL_DIR": "/content/softmeta_models/moss_voice_generator",
    "SOFTMETA_ECHOMIMIC_DIR": "/content/EchoMimicV3",
    "SOFTMETA_ECHOMIMIC_MODELS": "/content/echomimic_v3_models",
}

voice_python = env_python("moss312")
avatar_python = env_python("echo310")
if voice_python:
    env["SOFTMETA_VOICE_PYTHON"] = voice_python
if avatar_python:
    env["SOFTMETA_AVATAR_PYTHON"] = avatar_python

Path(env["HF_HOME"]).mkdir(parents=True, exist_ok=True)

log_handle = LOG.open("w", encoding="utf-8", errors="replace")
process = subprocess.Popen(
    [MM, "run", "-n", "sm311", "python", "-u", "start.py"],
    cwd=PROJECT,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
PID_FILE.write_text(str(process.pid), encoding="utf-8")

def port_open() -> bool:
    try:
        with socket.create_connection(("127.0.0.1", PORT), timeout=0.5):
            return True
    except OSError:
        return False

print("Starting SoftMeta Chatterbox TTS Server...")
for _ in range(300):
    if process.poll() is not None:
        log_handle.flush()
        raise RuntimeError(LOG.read_text(errors="replace")[-20000:])
    if port_open():
        break
    time.sleep(1)
else:
    raise TimeoutError("The server did not open port 8004. Run the log cell below.")

from urllib.request import urlopen
with urlopen(f"http://127.0.0.1:{PORT}/", timeout=15) as response:
    page_status = response.status
    page_preview = response.read(300).decode("utf-8", errors="replace")
if page_status != 200 or "<html" not in page_preview.lower():
    raise RuntimeError(f"Unexpected home-page response. HTTP {page_status}: {page_preview}")

from google.colab.output import eval_js
url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
display(HTML(
    '<p><a href="' + url + '" target="_blank" '
    'style="display:inline-block;padding:13px 19px;background:#5f52e8;color:#fff;'
    'border-radius:8px;text-decoration:none;font-weight:700">'
    'Open SoftMeta Audio and Avatar Studio</a></p>'
))
print("Home page check: HTTP 200 OK")
print("MOSS runtime:", "installed" if voice_python else "not installed")
print("Avatar runtime:", "installed" if avatar_python else "not installed")
print("Server PID:", process.pid)
print("Server log:", LOG)


## Recent server log


In [ ]:
from pathlib import Path
log = Path("/content/softmeta_chatterbox.log")
print(log.read_text(errors="replace")[-25000:] if log.exists() else "No server log yet.")


## Optional: Stop the server

Leave this disabled during normal use.


In [ ]:
STOP_SERVER = False  # @param {type:"boolean"}

import os
import signal
import subprocess
from pathlib import Path

pid_file = Path("/content/softmeta_chatterbox.pid")
if not STOP_SERVER:
    print("Server remains running.")
else:
    if pid_file.exists():
        try:
            os.kill(int(pid_file.read_text().strip()), signal.SIGTERM)
        except Exception:
            pass
        pid_file.unlink(missing_ok=True)
    subprocess.run("lsof -t -i:8004 | xargs -r kill -9", shell=True, check=False)
    print("Server stopped.")
